In [1]:
import pymysql

print("PyMySQL is working")

PyMySQL is working


In [2]:
import pandas as pd
from sqlalchemy import create_engine
from getpass import getpass

password = getpass("Enter MySQL password: ")

engine = create_engine(
    "mysql+pymysql://root:" + password + "@localhost/bingeplay"
)

print("Database connected successfully")

Enter MySQL password:  ········


Database connected successfully


In [3]:
query = "SELECT * FROM users LIMIT 5"

result = pd.read_sql(query, engine)

print(result)

  user_id           name signup_date       city age_group referral_source
0  U00001      Neha Bhat  2024-06-28  Bengaluru     18-24          friend
1  U00002    Kritika Roy  2024-03-31    Chennai     25-34    social_media
2  U00003  Ishita Shenoy  2024-06-08    Chennai     18-24    social_media
3  U00004  Kritika Menon  2024-01-04       Pune     25-34    social_media
4  U00005   Kiara Bansal  2024-02-12  Ahmedabad     35-44         organic


In [5]:
## Q1 – Active Revenue
##Find the number of active subscriptions and total monthly recurring revenue as of 30 June 2024.
query = """
SELECT
    COUNT(*) AS active_subscriptions,
    SUM(monthly_price_inr) AS monthly_revenue
FROM subscriptions
WHERE status = 'active'
AND (end_date IS NULL OR end_date > '2024-06-30');
"""

result = pd.read_sql(query, engine)

print(result)

   active_subscriptions  monthly_revenue
0                  2340         784260.0


In [6]:
## Q1 – Active Revenue

#Active Subscriptions: [2340]

#Total Monthly Recurring Revenue: ₹[784260.0]

#Interpretation:  
#As of 30 June 2024, BingePlay has [2340] active subscriptions generating ₹784260.0 in monthly recurring revenue.

In [8]:
 #Q2 
query = """
SELECT 
    MONTH(signup_date) AS month_number,
    COUNT(*) AS new_signups
FROM users
WHERE signup_date >= '2024-01-01'
AND signup_date < '2024-07-01'
GROUP BY MONTH(signup_date)
ORDER BY month_number;
"""

result = pd.read_sql(query, engine)

print(result)

highest_month = result.loc[result["new_signups"].idxmax()]

print("\nHighest Signup Month:")
print(highest_month)

   month_number  new_signups
0             1          350
1             2          400
2             3          500
3             4          550
4             5          600
5             6          600

Highest Signup Month:
month_number      5
new_signups     600
Name: 4, dtype: int64


In [9]:
## Q2 – Signup Momentum

#January Signups: [350]

#February Signups: [400]

#March Signups: [500]

#April Signups: [550]

#May Signups: [600]

#June Signups: [600]

#Highest Signup Month: [may]

#Highest Signup Count: [600]

#Interpretation:  The highest number of new users signed up in [may], with [600] signups.

In [10]:
#Q3
query = """
SELECT
    device_type,
    COUNT(*) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS average_watch_minutes,
    ROUND(
        100 * SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS completion_rate
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY device_type;
"""

result = pd.read_sql(query, engine)

print(result)

  device_type  total_sessions  total_watch_minutes  average_watch_minutes  \
0      Laptop           15105             453434.0                  30.02   
1      Mobile           50172            1504355.0                  29.98   
2      Tablet            7091             210733.0                  29.72   
3          TV           27981             840595.0                  30.04   

   completion_rate  
0            60.51  
1            60.24  
2            59.79  
3            59.98  


In [11]:
## Q3 – Device Analytics

#The query compares sessions, total watch time, average watch time per session, and completion rate for each device type.

#Interpretation:The results show how users watch BingePlay across different devices and which device has the highest completion rate.

In [13]:
#Q4
query = """
SELECT
    stars,
    COUNT(*) AS rating_count,
    ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM ratings), 2) AS percentage
FROM ratings
GROUP BY stars
ORDER BY stars;
"""

result = pd.read_sql(query, engine)

print(result)

four_five = pd.read_sql("""
SELECT
    ROUND(
        100 * SUM(CASE WHEN stars IN (4,5) THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS percentage_4_or_5
FROM ratings;
""", engine)

print("\nPercentage of 4 or 5 star ratings:")
print(four_five)

   stars  rating_count  percentage
0      1           234        4.68
1      2           352        7.04
2      3           847       16.94
3      4          1781       35.62
4      5          1786       35.72

Percentage of 4 or 5 star ratings:
   percentage_4_or_5
0              71.34


In [18]:
## Q4 – Rating Distribution

**1 Star: [234] ([4.68]%)

**2 Stars: [352] ([7.04]%)

**3 Stars: [847] ([16.94]%)

**4 Stars: [1781] ([35.62]%)

**5 Stars: [1786] ([35.72]%)

**Percentage of 4 or 5 Star Ratings: ([71.34]%)

#Interpretation:  A total of [71.34]% of all ratings are 4 or 5 stars.

SyntaxError: invalid syntax (1751582771.py, line 3)

In [19]:
#Q5
query = """
SELECT
    CASE
        WHEN is_original = 1 THEN 'Original'
        ELSE 'Acquired'
    END AS show_type,
    COUNT(*) AS number_of_shows,
    ROUND(AVG(imdb_rating), 2) AS average_imdb_rating,
    ROUND(AVG(release_year), 2) AS average_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original DESC;
"""

result = pd.read_sql(query, engine)

print(result)

  show_type  number_of_shows  average_imdb_rating  average_release_year
0  Original               30                 7.92               2020.37
1  Acquired               70                 6.63               2020.73


In [20]:
## Q5 – Originals vs Acquired

#| Show Type | Number of Shows | Average IMDb Rating | Average Release Year |
#|---|---:|---:|---:|
#| Original | [30] | [7.92] | [2020.37] |
#| Acquired | [70] | [6.63] | 2020.37 |

#Interpretation:  [Original/Acquired] shows have the higher average IMDb rating by [1.29] points.

In [21]:
query = """
WITH binge_days AS (
    SELECT
        user_id,
        show_id,
        DATE(session_date) AS watch_date,
        COUNT(*) AS session_count
    FROM watch_sessions
    WHERE session_date >= '2024-04-01'
      AND session_date < '2024-07-01'
      AND user_id IS NOT NULL
    GROUP BY user_id, show_id, DATE(session_date)
    HAVING COUNT(*) >= 5
),

user_binge_days AS (
    SELECT
        user_id,
        COUNT(*) AS binge_days
    FROM binge_days
    GROUP BY user_id
)

SELECT
    (SELECT COUNT(*) FROM binge_days) AS total_binge_days,
    user_id,
    binge_days
FROM user_binge_days
ORDER BY binge_days DESC
LIMIT 1;
"""

result = pd.read_sql(query, engine)

print(result)

   total_binge_days user_id  binge_days
0               414  U02956           8


In [22]:
## Q6 – Binge Day

**Total Binge Days: [414]

**User with Most Binge Days: [u02956]

**Binge Days for This User: [8]

**Interpretation:A binge day is counted when the same user watches the same show at least 5 times on the same date during Q2 2024.

SyntaxError: invalid syntax (3946693350.py, line 3)

In [23]:
#Q7
query = """
SELECT
    COUNT(*) AS total_q1_signups,
    SUM(
        CASE
            WHEN w.user_id IS NULL THEN 1
            ELSE 0
        END
    ) AS never_watched
FROM users u
LEFT JOIN (
    SELECT DISTINCT user_id
    FROM watch_sessions
    WHERE user_id IS NOT NULL
) w
ON u.user_id = w.user_id
WHERE u.signup_date >= '2024-01-01'
AND u.signup_date < '2024-04-01';
"""

result = pd.read_sql(query, engine)

print(result)

   total_q1_signups  never_watched
0              1250          226.0


In [24]:
## Q7 – Q1 Signups Who Never Watched

**Total Q1 Signups: [1250]

** Signups Who Never Watched: [226.0]

**Interpretation:  Among users who signed up during Q1 2024, [number] users had no watch session.

SyntaxError: invalid syntax (1890404418.py, line 3)

In [25]:
#Q8
query = """
WITH current_subscriptions AS (
    SELECT
        user_id,
        plan,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
    WHERE status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)

SELECT COUNT(*) AS over_paying_users
FROM current_subscriptions c
WHERE c.rn = 1
  AND c.plan IN ('Premium', 'Family')
  AND NOT EXISTS (
      SELECT 1
      FROM watch_sessions w
      JOIN shows s
        ON w.show_id = s.show_id
      WHERE w.user_id = c.user_id
        AND s.min_plan IN ('Premium', 'Family')
  );
"""

result = pd.read_sql(query, engine)

print(result)

   over_paying_users
0                212


In [29]:
## Q8 – Over-Paying- Users

##Number of Over-Paying Users: [212]

##Interpretation:  These Premium or Family users have only watched shows available on the Basic plan, so they may be paying for a higher plan
than their viewing behaviour requires.

SyntaxError: invalid syntax (196411839.py, line 6)

In [31]:
#Q9
query = """
WITH subscription_history AS (
    SELECT
        s.*,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date, subscription_id
        ) AS rn
    FROM subscriptions s
),

first_basic AS (
    SELECT
        user_id,
        start_date AS basic_start_date
    FROM subscription_history
    WHERE rn = 1
      AND plan = 'Basic'
),

first_upgrade AS (
    SELECT
        fb.user_id,
        MIN(s.start_date) AS upgrade_date
    FROM first_basic fb
    JOIN subscriptions s
      ON fb.user_id = s.user_id
    WHERE s.plan IN ('Premium', 'Family')
      AND s.start_date > fb.basic_start_date
    GROUP BY fb.user_id
),

active_users AS (
    SELECT DISTINCT user_id
    FROM subscriptions
    WHERE status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)

SELECT
    COUNT(*) AS upgrade_users,
    ROUND(
        AVG(DATEDIFF(fu.upgrade_date, u.signup_date)),
        2
    ) AS average_days_to_upgrade
FROM first_upgrade fu
JOIN users u
  ON fu.user_id = u.user_id
JOIN active_users a
  ON fu.user_id = a.user_id
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-02-01';
"""

result = pd.read_sql(query, engine)

print(result)

   upgrade_users  average_days_to_upgrade
0             55                    64.96


In [32]:
## Q9 – Upgrade Success Cohort

**Number of Successful Upgrade Users: [55]

**Average Days from Signup to First Upgrade: 64.96 days

**Interpretation:These January 2024 users started with Basic, later upgraded to Premium or Family, and remained active as of 30 June 2024.

SyntaxError: invalid syntax (2812992551.py, line 3)

In [33]:
#Q10
query = """
WITH comeback_events AS (
    SELECT DISTINCT
        w1.user_id,
        w1.show_id,
        DATE(w1.session_date) AS incomplete_date
    FROM watch_sessions w1
    JOIN watch_sessions w2
      ON w1.user_id = w2.user_id
     AND w1.show_id = w2.show_id
     AND DATE(w2.session_date) > DATE(w1.session_date)
     AND DATE(w2.session_date) <= DATE(w1.session_date) + INTERVAL 7 DAY
    WHERE w1.completed = 0
      AND w1.user_id IS NOT NULL
)

SELECT
    COUNT(*) AS total_comeback_events,
    (
        SELECT show_id
        FROM comeback_events
        GROUP BY show_id
        ORDER BY COUNT(*) DESC, show_id
        LIMIT 1
    ) AS top_show_id,
    (
        SELECT s.title
        FROM shows s
        WHERE s.show_id = (
            SELECT show_id
            FROM comeback_events
            GROUP BY show_id
            ORDER BY COUNT(*) DESC, show_id
            LIMIT 1
        )
    ) AS top_show_title
FROM comeback_events;
"""

result = pd.read_sql(query, engine)

print(result)

   total_comeback_events top_show_id    top_show_title
0                   4345        S088  Rayalaseema Raga


In [34]:
## Q10 – Cliffhanger Comebacks

**Total Comeback Events: [4345]

**Show ID with Most Events: [5088]

**Show Title: [Rayalaseema Raga]

**Interpretation:  A comeback event occurs when a user has an incomplete session and watches the same show again within the next 1–7 days.

SyntaxError: invalid character '–' (U+2013) (3650183269.py, line 9)

In [35]:
#Q11
query = """
WITH weekly_activity AS (
    SELECT DISTINCT
        user_id,
        YEARWEEK(session_date, 3) AS week_key
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),

numbered AS (
    SELECT
        user_id,
        week_key,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY week_key
        ) AS rn
    FROM weekly_activity
),

streaks AS (
    SELECT
        user_id,
        week_key,
        week_key - rn AS grp
    FROM numbered
),

streak_lengths AS (
    SELECT
        user_id,
        grp,
        COUNT(*) AS streak_length
    FROM streaks
    GROUP BY user_id, grp
),

user_longest AS (
    SELECT
        user_id,
        MAX(streak_length) AS longest_streak
    FROM streak_lengths
    GROUP BY user_id
)

SELECT
    COUNT(CASE WHEN longest_streak >= 4 THEN 1 END) AS users_with_4_week_streak,
    MAX(longest_streak) AS longest_streak,
    (
        SELECT user_id
        FROM user_longest
        ORDER BY longest_streak DESC, user_id
        LIMIT 1
    ) AS user_with_longest_streak
FROM user_longest;
"""

result = pd.read_sql(query, engine)

print(result)

   users_with_4_week_streak  longest_streak user_with_longest_streak
0                      1675              26                   U00213


In [36]:
## Q11 – Consecutive-Week Engagement

**Users with 4+ Consecutive Weeks: 1675

**Longest Streak: [26] weeks

**One User with Longest Streak: [u00213]

**Interpretation: The gaps-and-islands method was used to identify users who had at least one watch session in four or more consecutive 
ISO calendar weeks.

SyntaxError: invalid syntax (2380634097.py, line 3)

In [37]:
#Q12
query = """
WITH monthly_watch AS (
    SELECT
        user_id,
        SUM(
            CASE
                WHEN session_date >= '2024-05-01'
                 AND session_date < '2024-06-01'
                THEN watch_minutes
                ELSE 0
            END
        ) AS may_minutes,

        SUM(
            CASE
                WHEN session_date >= '2024-06-01'
                 AND session_date < '2024-07-01'
                THEN watch_minutes
                ELSE 0
            END
        ) AS june_minutes
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date >= '2024-05-01'
      AND session_date < '2024-07-01'
    GROUP BY user_id
),

churn_signal AS (
    SELECT
        u.user_id,
        u.name,
        m.may_minutes,
        m.june_minutes,
        ROUND(
            100 * (m.may_minutes - m.june_minutes)
            / m.may_minutes,
            2
        ) AS drop_percentage
    FROM monthly_watch m
    JOIN users u
      ON m.user_id = u.user_id
    WHERE m.may_minutes > 0
      AND m.june_minutes <= m.may_minutes * 0.5
)

SELECT *
FROM churn_signal
ORDER BY drop_percentage DESC;
"""

result = pd.read_sql(query, engine)

print(result)

print("\nTotal Churn Signal Users:", len(result))

    user_id              name  may_minutes  june_minutes  drop_percentage
0    U00023    Amit Mukherjee         43.0           0.0           100.00
1    U00166  Shaurya Malhotra         94.0           0.0           100.00
2    U00211        Ravi Menon        336.0           0.0           100.00
3    U00225     Kritika Patil        209.0           0.0           100.00
4    U00237    Shaurya Bansal        126.0           0.0           100.00
..      ...               ...          ...           ...              ...
516  U01858  Sanjay Mukherjee        296.0         147.0            50.34
517  U02530       Nandini Roy        451.0         224.0            50.33
518  U01192   Rohit Mukherjee        136.0          68.0            50.00
519  U01806      Yuvraj Raman         78.0          39.0            50.00
520  U02100      Vikram Singh        204.0         102.0            50.00

[521 rows x 5 columns]

Total Churn Signal Users: 521


In [ ]:
## Q12 – Churn Signal

**Total Churn Signal Users: [512]

**The users below experienced a drop of at least 50% in watch minutes from May to June 2024.

**| User ID | Name | May Minutes | June Minutes | Drop % |
**|---|---|---:|---:|---:|
**| U00023] | [answer] | [answer] | [answer] | [answer]% |

Interpretation:  
These users show a significant decline in watch activity and can be treated as potential churn-risk users.